In [1]:
import sys, os, shutil, torch

CODE = "/kaggle/input/datasets/ashwanisingh997/thesis-code-1229692"
IMG  = "/kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692"
SPL  = "/kaggle/input/datasets/ashwanisingh997/ham10000-splits-1229692"
WORK = "/kaggle/working/repo"

os.makedirs(f"{WORK}/src", exist_ok=True)
os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    dst = f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}"
    shutil.copy(f"{CODE}/{f}", dst)

sys.path.insert(0, WORK)
os.chdir(WORK)

print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="3853a1a")

2.10.0+cu128 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.10.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud


In [2]:
from src.data import HAM10000, make_loader, build_image_index, assert_no_synthetic_leakage

assert_no_synthetic_leakage(f"{SPL}/val.csv", f"{SPL}/test.csv")

index = build_image_index(IMG)

NW = 4   # Kaggle is Linux; Run 2 uses 0 because Windows Jupyter hangs with workers

val_ds  = HAM10000(f"{SPL}/val.csv",  IMG, size=128, augment=False, index=index)
test_ds = HAM10000(f"{SPL}/test.csv", IMG, size=128, augment=False, index=index)
val_loader  = make_loader(val_ds,  batch_size=32, shuffle=False, num_workers=NW)
test_loader = make_loader(test_ds, batch_size=32, shuffle=False, num_workers=NW)

print(f"val {len(val_ds)}  test {len(test_ds)}")

[data] no synthetic leakage in 2 partition(s)
[data] indexed 10015 images under /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
val 1532  test 1502


In [3]:
from src.utils import set_seed
from src.classifier import build_model, train_classifier
from src.metrics import evaluate, save_metrics

results_run1_A = {}

for seed in [42, 43, 44]:
    print(f"\n{'='*60}\nRUN 1 — CONDITION A — SEED {seed}\n{'='*60}")
    set_seed(seed)

    tr = HAM10000(f"{SPL}/train.csv", IMG, size=128, augment=False, index=index)
    tr_ld = make_loader(tr, batch_size=32, shuffle=True, num_workers=NW)

    m, hist = train_classifier(
        build_model(), tr_ld, val_loader,
        epochs=30, lr=1e-4, weight_decay=1e-4, patience=7,
        tag=f"A_baseline_s{seed}", log_dir=f"{root}/logs",
    )
    r = evaluate(m, test_loader, bootstrap=True, seed=seed)
    save_metrics(r, f"A_baseline_seed{seed}", root=str(root))
    results_run1_A[seed] = r


RUN 1 — CONDITION A — SEED 42
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s] 


ep 00  loss 0.7981  val macroF1 0.4772  val acc 0.7774
ep 01  loss 0.3912  val macroF1 0.5509  val acc 0.8055
ep 02  loss 0.1783  val macroF1 0.5840  val acc 0.7892
ep 03  loss 0.1147  val macroF1 0.6222  val acc 0.7983
ep 04  loss 0.0567  val macroF1 0.6558  val acc 0.8087
ep 05  loss 0.0467  val macroF1 0.6207  val acc 0.8022
ep 06  loss 0.0355  val macroF1 0.6121  val acc 0.8061
ep 07  loss 0.0377  val macroF1 0.6402  val acc 0.8048
ep 08  loss 0.0254  val macroF1 0.6508  val acc 0.8120
ep 09  loss 0.0245  val macroF1 0.6532  val acc 0.8159
ep 10  loss 0.0171  val macroF1 0.6409  val acc 0.8166
ep 11  loss 0.0091  val macroF1 0.6327  val acc 0.8120
Early stopping at epoch 11 (best macro-F1 0.6558)
              precision    recall  f1-score   support

       akiec      0.524     0.423     0.468        52
         bcc      0.593     0.676     0.632        71
         bkl      0.709     0.569     0.631       167
          df      0.381     0.400     0.390        20
         mel      0

In [4]:
import numpy as np, pandas as pd, json

seeds = [42, 43, 44]
METRICS = ["macro_f1", "melanoma_recall", "balanced_accuracy", "accuracy", "macro_auroc"]

rows = [{"metric": m,
         "mean": np.mean([results_run1_A[s][m] for s in seeds]),
         "sd":   np.std([results_run1_A[s][m] for s in seeds], ddof=1),
         "values": [round(results_run1_A[s][m], 4) for s in seeds]}
        for m in METRICS]
summ = pd.DataFrame(rows).round(4)
print(summ.to_string(index=False))
summ.to_csv(f"{root}/metrics/A_baseline_summary.csv", index=False)

import subprocess
print(subprocess.getoutput(f"find {root} -type f | sort"))

           metric   mean     sd                   values
         macro_f1 0.6220 0.0170 [0.6053, 0.6215, 0.6392]
  melanoma_recall 0.4251 0.0452 [0.4731, 0.3832, 0.4192]
balanced_accuracy 0.5955 0.0157 [0.5853, 0.5875, 0.6136]
         accuracy 0.7994 0.0037 [0.8029, 0.7956, 0.7996]
      macro_auroc 0.9293 0.0107 [0.9319, 0.9175, 0.9384]
/kaggle/working/results/run1_cloud/logs/A_baseline_s42_history.json
/kaggle/working/results/run1_cloud/logs/A_baseline_s43_history.json
/kaggle/working/results/run1_cloud/logs/A_baseline_s44_history.json
/kaggle/working/results/run1_cloud/metrics/A_baseline_seed42.json
/kaggle/working/results/run1_cloud/metrics/A_baseline_seed43.json
/kaggle/working/results/run1_cloud/metrics/A_baseline_seed44.json
/kaggle/working/results/run1_cloud/metrics/A_baseline_summary.csv
/kaggle/working/results/run1_cloud/run_meta.json


In [5]:
results_run1_B = {}

for seed in [42, 43, 44]:
    print(f"\n{'='*60}\nRUN 1 — CONDITION B — SEED {seed}\n{'='*60}")
    set_seed(seed)

    tr = HAM10000(f"{SPL}/train.csv", IMG, size=128, augment=True, index=index)
    tr_ld = make_loader(tr, batch_size=32, shuffle=True, num_workers=NW)

    m, hist = train_classifier(
        build_model(), tr_ld, val_loader,
        epochs=30, lr=1e-4, weight_decay=1e-4, patience=7,
        tag=f"B_classical_s{seed}", log_dir=f"{root}/logs",
    )
    r = evaluate(m, test_loader, bootstrap=True, seed=seed)
    save_metrics(r, f"B_classical_seed{seed}", root=str(root))
    results_run1_B[seed] = r


RUN 1 — CONDITION B — SEED 42
ep 00  loss 0.8485  val macroF1 0.4551  val acc 0.7833
ep 01  loss 0.5817  val macroF1 0.5591  val acc 0.8114
ep 02  loss 0.4969  val macroF1 0.5864  val acc 0.7970
ep 03  loss 0.4372  val macroF1 0.5975  val acc 0.7983
ep 04  loss 0.3919  val macroF1 0.6115  val acc 0.8042
ep 05  loss 0.3564  val macroF1 0.6586  val acc 0.8035
ep 06  loss 0.3085  val macroF1 0.6468  val acc 0.8048
ep 07  loss 0.2796  val macroF1 0.6393  val acc 0.8048
ep 08  loss 0.2565  val macroF1 0.6362  val acc 0.8035
ep 09  loss 0.2341  val macroF1 0.6498  val acc 0.8068
ep 10  loss 0.1971  val macroF1 0.6568  val acc 0.8120
ep 11  loss 0.1940  val macroF1 0.6435  val acc 0.8055
ep 12  loss 0.1771  val macroF1 0.6472  val acc 0.7990
Early stopping at epoch 12 (best macro-F1 0.6586)
              precision    recall  f1-score   support

       akiec      0.532     0.635     0.579        52
         bcc      0.716     0.676     0.696        71
         bkl      0.639     0.509     0.5

In [7]:
import numpy as np, pandas as pd, json

seeds = [42, 43, 44]
METRICS = ["macro_f1", "melanoma_recall", "balanced_accuracy", "accuracy", "macro_auroc"]

rows = []
for m in METRICS:
    a = [results_run1_A[s][m] for s in seeds]
    b = [results_run1_B[s][m] for s in seeds]
    rows.append({"metric": m,
                 "A_mean": np.mean(a), "A_sd": np.std(a, ddof=1),
                 "B_mean": np.mean(b), "B_sd": np.std(b, ddof=1),
                 "diff": np.mean(b) - np.mean(a)})
comp = pd.DataFrame(rows).round(4)
print("RUN 1 (Kaggle T4)")
print(comp.to_string(index=False))
comp.to_csv(f"{root}/metrics/AB_comparison.csv", index=False)

summ = pd.DataFrame([{"metric": m,
                      "mean": np.mean([results_run1_B[s][m] for s in seeds]),
                      "sd": np.std([results_run1_B[s][m] for s in seeds], ddof=1),
                      "values": [round(results_run1_B[s][m], 4) for s in seeds]}
                     for m in METRICS]).round(4)
summ.to_csv(f"{root}/metrics/B_classical_summary.csv", index=False)

print("\nPer-class recall (A -> B), Run 1")
from src.data import CLASSES
for c in CLASSES:
    a = [results_run1_A[s]["per_class"][c]["recall"] for s in seeds]
    b = [results_run1_B[s]["per_class"][c]["recall"] for s in seeds]
    n = results_run1_A[42]["per_class"][c]["support"]
    print(f"  {c:6s} {np.mean(a):.3f}±{np.std(a,ddof=1):.3f} -> "
          f"{np.mean(b):.3f}±{np.std(b,ddof=1):.3f}  diff {np.mean(b)-np.mean(a):+.3f}  (n={n})")

RUN 1 (Kaggle T4)
           metric  A_mean   A_sd  B_mean   B_sd   diff
         macro_f1  0.6220 0.0170  0.6636 0.0150 0.0416
  melanoma_recall  0.4251 0.0452  0.5070 0.1006 0.0818
balanced_accuracy  0.5955 0.0157  0.6461 0.0302 0.0507
         accuracy  0.7994 0.0037  0.8105 0.0034 0.0111
      macro_auroc  0.9293 0.0107  0.9505 0.0024 0.0212

Per-class recall (A -> B), Run 1
  akiec  0.526±0.091 -> 0.590±0.040  diff +0.064  (n=52)
  bcc    0.653±0.053 -> 0.690±0.037  diff +0.038  (n=71)
  bkl    0.561±0.019 -> 0.507±0.009  diff -0.054  (n=167)
  df     0.417±0.029 -> 0.450±0.132  diff +0.033  (n=20)
  mel    0.425±0.045 -> 0.507±0.101  diff +0.082  (n=167)
  nv     0.937±0.002 -> 0.938±0.020  diff +0.001  (n=1004)
  vasc   0.651±0.027 -> 0.841±0.073  diff +0.190  (n=21)
